# Google Earth Engine: Downloader ERA5-Land Hourly (Multi-Band Stacking to NetCDF)
Notebook ini digunakan untuk mengunduh data curah hujan reanalisis **ERA5-Land Hourly** (`ECMWF/ERA5_LAND/HOURLY`) dari Google Earth Engine secara presisi dan super cepat menggunakan teknik *Multi-Band Stacking* bulanan.

### Fitur Utama:
- **Konversi Otomatis**: Mengubah satuan curah hujan dari meter (m) menjadi milimeter (mm) dengan perkalian `x1000`.
- **Multi-Band Stacking**: Mengunduh 1 bulan jam-jaman (~744 jam) hanya dalam 1 *request* tunggal.
- **Kaggle Compatible**: Autentikasi otomatis menggunakan `GEE_KEY` dari Kaggle Secrets.

In [ ]:
!pip install earthengine-api rioxarray geopandas xarray requests

In [ ]:
import os
import json
import time
import requests
import zipfile
import io
import ee
import rioxarray as rxr
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
from google.oauth2.service_account import Credentials
from kaggle_secrets import UserSecretsClient

# Autentikasi GEE via Kaggle Secrets (GEE_KEY)
try:
    user_secrets = UserSecretsClient()
    service_account_info = json.loads(user_secrets.get_secret("GEE_KEY"))
    SCOPES = ['https://www.googleapis.com/auth/earthengine']
    credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
    ee.Initialize(credentials=credentials, project='staklimjerukagung')
    print("✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)")
except Exception as e:
    print(f"⚠️ Gagal Inisialisasi via Secrets, mencoba auth manual: {e}")
    ee.Authenticate()
    ee.Initialize(project='staklimjerukagung')

In [ ]:
# Parameter Wilayah (Bounding Box Kebumen) & Direktori Output
minx, miny, maxx, maxy = 109.3, -7.9, 110.0, -7.4
geometry = ee.Geometry.BBox(minx, miny, maxx, maxy)

folder_induk = Path("data/era5_land")
folder_induk.mkdir(parents=True, exist_ok=True)

tahun_awal = 2005
tahun_akhir = 2026

print(f"Batas Wilayah (Bounding Box Kebumen): Min Lon: {minx}, Min Lat: {miny}, Max Lon: {maxx}, Max Lat: {maxy}")
print(f"Folder Output: {folder_induk.absolute()}")

In [ ]:
def download_era5_land_month(year, month):
    out_dir = folder_induk / str(year)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    nc_path = out_dir / f"era5_land_{year}_{month:02d}.nc"
    if nc_path.exists():
        print(f"ℹ️ File sudah ada: {nc_path.name}, melewati...")
        return
        
    start_date = f"{year}-{month:02d}-01"
    if month == 12:
        end_date = f"{year+1}-01-01"
    else:
        end_date = f"{year}-{month+1:02d}-01"
        
    print(f"\nProcessing ERA5-Land: {year}-{month:02d} ({start_date} s.d. {end_date})...")
    
    # Filter ERA5-Land Hourly collection & konversi m -> mm (* 1000)
    col = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
           .filterBounds(geometry)
           .filterDate(start_date, end_date)
           .select("total_precipitation_hourly"))
    
    count = col.size().getInfo()
    if count == 0:
        print(f"⚠️ Data tidak tersedia untuk {year}-{month:02d}")
        return
        
    # Ekstrak timestamp presisi dari metadata GEE
    timestamps = col.aggregate_array("system:time_start").getInfo()
    dates = pd.to_datetime(timestamps, unit='ms')
    
    # Kalikan 1000 untuk mengubah m -> mm
    col_mm = col.map(lambda img: img.multiply(1000).copyProperties(img, ["system:time_start"]))
    
    # Multi-band stacking
    stacked_img = col_mm.toBands()
    
    # Dapatkan URL unduhan GeoTIFF Zip
    url = stacked_img.getDownloadURL({
        'name': f"era5_land_{year}_{month:02d}",
        'region': geometry,
        'scale': 11132,
        'format': 'GEO_TIFF'
    })
    
    # Unduh zip file
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    
    z = zipfile.ZipFile(io.BytesIO(r.content))
    tif_filename = z.namelist()[0]
    temp_tif = f"temp_era5_{year}_{month:02d}.tif"
    z.extract(tif_filename, path=".")
    os.rename(tif_filename, temp_tif)
    
    # Buka TIF dengan rioxarray & ubah ke NetCDF 3D (time, y, x)
    da = rxr.open_rasterio(temp_tif, masked=True)
    if "band" in da.dims:
        da = da.rename({"band": "time"})
    da = da.assign_coords(time=dates)
    da.name = "precipitation"
    
    # Simpan sebagai NetCDF
    da.to_netcdf(nc_path)
    
    da.close()
    if os.path.exists(temp_tif):
        os.remove(temp_tif)
        
    print(f"✅ Selesai ({count} jam data tersimpan): {nc_path.name}")

# Eksekusi Loop
for year in range(tahun_awal, tahun_akhir + 1):
    for month in range(1, 13):
        try:
            download_era5_land_month(year, month)
        except Exception as e:
            print(f"❌ Error {year}-{month:02d}: {e}")